# 🚀 Day 29/30 — Handwritten Digit Recognition

## Mini Machine Learning Project

This project builds a handwritten digit classification system using Scikit-learn's built-in **Digits dataset**.

**Objective:** Predict handwritten digits from **0 to 9** using Logistic Regression, KNN, Random Forest, and SVM, then select the best-performing model.

**Workflow:** Data Loading → Data Understanding → Visualization → Train/Test Split → Model Training → Evaluation → Cross-Validation → Model Selection → Confusion Matrix → Model Persistence → Prediction


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

print("Libraries imported successfully!")


## 1. Load the Built-in Digits Dataset

In [ ]:
digits = load_digits()

X = pd.DataFrame(
    digits.data,
    columns=[f"pixel_{i}" for i in range(digits.data.shape[1])]
)
y = pd.Series(digits.target, name="digit")

print("Dataset loaded successfully!")
print(f"Number of samples : {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes : {len(np.unique(y))}")
print("Classes:", sorted(y.unique()))


## 2. Data Understanding

In [ ]:
print("First 5 rows:")
display(X.head())

print("\nTarget distribution:")
display(y.value_counts().sort_index())

print("\nMissing values:", X.isnull().sum().sum())
print("Duplicate rows:", X.duplicated().sum())


## 3. Visualize Sample Handwritten Digits

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for ax, image, label in zip(axes.ravel(), digits.images[:10], digits.target[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Digit: {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 4. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples : {X_test.shape[0]}")


## 5. Create Machine Learning Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=42))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, random_state=42))
    ])
}

print("Models created:")
for name in models:
    print("-", name)


## 6. Train and Evaluate Models

In [ ]:
results = []
trained_models = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })
    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(
    by="Accuracy", ascending=False
).reset_index(drop=True)

display(results_df)


## 7. Model Comparison

In [ ]:
results_df.to_csv("Day29_Model_Results.csv", index=False)
print("Results saved as Day29_Model_Results.csv")

plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.title("Model Accuracy Comparison")
plt.ylim(0.90, 1.00)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 8. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for model_name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )
    cv_results.append({
        "Model": model_name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values(
    by="CV Mean Accuracy", ascending=False
).reset_index(drop=True)

display(cv_df)


## 9. Select the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)


## 10. Classification Report

In [ ]:
best_predictions = best_model.predict(X_test)

print(classification_report(
    y_test, best_predictions, zero_division=0
))


## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_predictions)
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=digits.target_names
)
disp.plot()
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 12. Save the Best Model

In [ ]:
MODEL_FILE = "digit_model.pkl"
joblib.dump(best_model, MODEL_FILE)
print(f"Best model saved successfully as: {MODEL_FILE}")


## 13. Load the Saved Model

In [ ]:
loaded_model = joblib.load(MODEL_FILE)
print("Saved model loaded successfully!")


## 14. Test a Sample Prediction

In [ ]:
sample_index = 0
sample = X_test.iloc[[sample_index]]
actual_digit = y_test.iloc[sample_index]
prediction = loaded_model.predict(sample)[0]

print("Actual digit    :", actual_digit)
print("Predicted digit :", prediction)


## 15. Prediction Probabilities

In [ ]:
if hasattr(loaded_model, "predict_proba"):
    probabilities = loaded_model.predict_proba(sample)[0]
    probability_df = pd.DataFrame({
        "Digit": range(10),
        "Probability": probabilities
    }).sort_values("Probability", ascending=False)
    display(probability_df)


## 📊 Results

In the completed run, **SVM achieved 97.50% test accuracy** and approximately **98.12% mean cross-validation accuracy**, making it the best-performing model among the four tested models.

The confusion matrix shows that most digits were classified correctly.

## 💡 Key Learnings

1. Working with an image classification dataset.
2. Representing 8×8 images as 64 numerical pixel features.
3. Comparing multiple classification algorithms.
4. Understanding why scaling is useful for Logistic Regression, KNN, and SVM.
5. Using cross-validation to estimate model stability.
6. Interpreting a multiclass confusion matrix.
7. Saving and loading an ML model with Joblib.
8. Generating predictions and class probabilities.

# ✅ Day 29 Complete
**Handwritten Digit Recognition — Mini Machine Learning Project**
